# 04 - Fairness: group diagnostics, fair models, proxy check

The target is a lender's recorded decision, not creditworthiness or default. So every number below says how a model behaves against past decisions, group by group. Agreeing with those decisions is not the same as being right about the applicant. All numbers are associations, not causes.

Three parts:

1. The competition model (the final blend), group by group.
2. Four fair models in a 2 x 2: two feature lists (the Kaggle list, and the clean list without the two leaks) crossed with two treatments (race-blind, and race-blind with reweighing). For each: what it costs in AUC, and how the gaps between groups move.
3. The proxy check: how well the columns of a race-blind model predict an applicant's race.

Every part only reads files already written by "make cv" and "src/proxy_check.py": nothing is trained here. The bootstrap of part 2 takes a few minutes.

In [20]:
import sys
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from compare import fold_diff, weighted_auc
from data import load
from features import BLOCKS, build

RUN_ID = "20261031T220901Z_1ff742f2"   # final blend: LightGBM 2/6 + CatBoost 2/6 + ExtraTrees 1/6 + XGBoost 1/6
N_BOOT, BOOT_SEED = 1000, 0            # same as src/compare.py
MIN_N = 500                            # under this, a group is shown but not read

train, test = load()
runs = pd.read_csv(ROOT / "results" / "runs.csv")


def oof_pred(run_id):
    """Out-of-fold predictions of a logged run, in the row order of train."""
    o = pd.read_parquet(ROOT / runs.set_index("run_id").oof_path[run_id])
    m = train[["id", "accepted"]].merge(o, on="id", validate="1:1", suffixes=("", "_oof"))
    assert (m.accepted == m.accepted_oof).all()
    return m.pred.to_numpy(), m.fold.to_numpy()


df = train
p, fold = oof_pred(RUN_ID)
y = df.accepted.to_numpy()
print(len(df), "rows   pooled AUC", round(roc_auc_score(y, p), 5), "  accepted", round(y.mean(), 4), "  mean prediction", round(p.mean(), 4))

105000 rows   pooled AUC 0.92043   accepted 0.8499   mean prediction 0.8516


## Choices

Groups: "derived_race", "derived_ethnicity" and "derived_sex" (they summarise applicant and co-applicant together), and "applicant_age". They are the official HMDA summaries, so a reader can check them. "Joint", "Not Available" and "Free Form Text Only" are kept: dropping them would hide a large share of the rows.

Reference group: the largest group with a single declared value (White, Not Hispanic or Latino, Male, 25-34). A gap is measured against it because a gap needs a base, not because that group is the norm. "Joint" is larger for sex, but it describes a couple, not a person.

Two cut-offs, set for each model on its own predictions: the metric is AUC, so there is no real decision threshold and any choice is a convention. We take 0.50 (the plain reading of a probability) and the cut-off that accepts as many rows as were really accepted (0.85 of them). The same cut-off applies to every group: separate cut-offs would be a different model.

Positive class = accepted. So TPR = share of accepted rows the model also accepts, FPR = share of denied rows the model accepts anyway. Here "error" means "disagrees with the lender".

Intervals: bootstrap of rows inside each group, 1000 resamples, percentile 95%. The group sizes stay fixed. A gap's interval takes the two groups' resamples pair by pair. These intervals cover the sampling of rows, not the training of the model, and they are not corrected for reading many groups at once.

"AUC without leak rows" recomputes AUC without the rows where a confirmed leak is active (preapproval = 1, initially_payable_to_institution = 3).

In [2]:
GROUPS = {"derived_race": "White", "derived_ethnicity": "Not Hispanic or Latino",
          "derived_sex": "Male", "applicant_age": "25-34"}
leak = (df.preapproval.astype("string") == "1") | (df.initially_payable_to_institution.astype("string") == "3")
keep = ~leak.to_numpy()
STATS = ["observed", "predicted", "pred_minus_obs", "auc", "auc_no_leak"]
for c in ("0.50", "same rate"):
    STATS += [f"accept {c}", f"tpr {c}", f"fpr {c}"]


def cut_offs(p):
    return {"0.50": 0.5, "same rate": float(np.quantile(p, 1 - y.mean()))}


def group_stats(y, p, keep, cuts):
    """Return f(w): every statistic of one group for row weights w (a resample = how many times each row is drawn)."""
    auc, auc_kept = weighted_auc(y, p), weighted_auc(y[keep], p[keep])
    above = [p >= t for t in cuts.values()]

    def f(w):
        total, pos, neg = w.sum(), (w * y).sum(), (w * (1 - y)).sum()
        obs, pred = pos / total, (w * p).sum() / total
        out = [obs, pred, pred - obs, auc(w), auc_kept(w[keep])]
        for a in above:
            out += [(w * a).sum() / total, (w * y * a).sum() / pos, (w * (1 - y) * a).sum() / neg]
        return np.array(out)

    return f


def diagnostics(p, attr, levels=None):
    """(point, draws, size) per group of attr: point estimates, bootstrap draws, (n, denied)."""
    rng, cuts = np.random.default_rng(BOOT_SEED), cut_offs(p)
    labels = df[attr].astype("string").fillna("(blank)").to_numpy()
    point, draws, size = {}, {}, {}
    for level in levels or pd.Series(labels).value_counts().index:
        i = np.flatnonzero(labels == level)
        f, n = group_stats(y[i], p[i], keep[i], cuts), len(i)
        point[level] = f(np.ones(n))
        draws[level] = np.array([f(np.bincount(rng.integers(0, n, n), minlength=n).astype(float)) for _ in range(N_BOOT)])
        size[level] = (n, int((y[i] == 0).sum()))
    return point, draws, size


np.seterr(all="ignore")   # very small groups can draw a resample with no denied row: that draw is NaN
results = {attr: diagnostics(p, attr) for attr in GROUPS}
print("cut-offs", {k: round(v, 4) for k, v in cut_offs(p).items()}, "  leak rows", int(leak.sum()))

cut-offs {'0.50': 0.5, 'same rate': 0.7797}   leak rows 6279


In [3]:
# Checks: the weighted formulas must give back the plain ones.
i = np.flatnonzero(df.derived_race.astype("string") == "Black or African American")
d = dict(zip(STATS, results["derived_race"][0]["Black or African American"]))
assert abs(d["auc"] - roc_auc_score(y[i], p[i])) < 1e-9
assert abs(d["auc_no_leak"] - roc_auc_score(y[i][keep[i]], p[i][keep[i]])) < 1e-9
assert abs(d["observed"] - y[i].mean()) < 1e-12
t = cut_offs(p)["same rate"]
assert abs(d["tpr same rate"] - (p[i][y[i] == 1] >= t).mean()) < 1e-12
assert abs(d["fpr same rate"] - (p[i][y[i] == 0] >= t).mean()) < 1e-12
print("checks passed")

checks passed


## 1. Competition model: calibration and ranking inside each group

Observed = share accepted. Predicted = mean predicted probability. A positive gap means the model expects more acceptances than the lender gave. Groups under 500 rows are marked and not read.

In [4]:
def ci(a, digits=3):
    lo, hi = np.nanquantile(a, [0.025, 0.975])
    return f"[{lo:.{digits}f}, {hi:.{digits}f}]"


def table_one(res, attr):
    point, draws, size = res[attr]
    rows = []
    for level, pt in point.items():
        n, denied = size[level]
        row = {"group": level, "n": n, "denied": denied}
        if n < MIN_N:
            rows.append({**row, "read": "no, n < 500"})
            continue
        d, b = dict(zip(STATS, pt)), draws[level]
        col = lambda s: b[:, STATS.index(s)]
        row |= {"observed": f"{d['observed']:.3f} {ci(col('observed'))}",
                "predicted": f"{d['predicted']:.3f}",
                "pred - obs": f"{d['pred_minus_obs']:+.3f} {ci(col('pred_minus_obs'))}",
                "AUC": f"{d['auc']:.3f} {ci(col('auc'))}",
                "AUC without leak rows": f"{d['auc_no_leak']:.3f} {ci(col('auc_no_leak'))}", "read": "yes"}
        rows.append(row)
    return pd.DataFrame(rows).fillna("")


for attr in GROUPS:
    print(attr); print(table_one(results, attr).to_string(index=False)); print()

derived_race
                                    group     n  denied             observed predicted             pred - obs                  AUC AUC without leak rows        read
                                    White 66107    8721 0.868 [0.865, 0.871]     0.869 +0.001 [-0.001, 0.002] 0.921 [0.917, 0.924]  0.907 [0.903, 0.911]         yes
                       Race Not Available 17872    3082 0.828 [0.822, 0.833]     0.830 +0.003 [-0.001, 0.006] 0.923 [0.916, 0.929]  0.909 [0.901, 0.916]         yes
                Black or African American  8991    2349 0.739 [0.730, 0.748]     0.744 +0.005 [-0.001, 0.011] 0.909 [0.901, 0.917]  0.896 [0.886, 0.904]         yes
                                    Asian  8009     888 0.889 [0.882, 0.896]     0.893 +0.004 [-0.001, 0.008] 0.882 [0.867, 0.896]  0.855 [0.837, 0.872]         yes
                                    Joint  2716     347 0.872 [0.859, 0.884]     0.871 -0.001 [-0.009, 0.007] 0.911 [0.889, 0.933]  0.895 [0.870, 0.921]         y

## 1. Competition model: decisions at a fixed cut-off, gaps against the reference group

For each group: share the model accepts, TPR and FPR, then the two gaps against the reference group with their 95% intervals. A gap whose interval excludes 0 is a measured difference on these rows; it does not say why.

In [5]:
def table_two(res, attr, cut):
    point, draws, size = res[attr]
    ref, rows = GROUPS[attr], []
    for level, pt in point.items():
        n = size[level][0]
        row = {"group": level + (" (reference)" if level == ref else ""), "n": n}
        if n < MIN_N:
            rows.append({**row, "read": "no, n < 500"})
            continue
        d = dict(zip(STATS, pt))
        row |= {"accepts": f"{d[f'accept {cut}']:.3f}", "TPR": f"{d[f'tpr {cut}']:.3f}", "FPR": f"{d[f'fpr {cut}']:.3f}"}
        if level != ref:
            for s, name in ((f"tpr {cut}", "TPR gap"), (f"fpr {cut}", "FPR gap")):
                k = STATS.index(s)
                row[name] = f"{pt[k] - point[ref][k]:+.3f} {ci(draws[level][:, k] - draws[ref][:, k])}"
        rows.append({**row, "read": "yes"})
    return pd.DataFrame(rows).fillna("")


for cut, t in cut_offs(p).items():
    for attr in GROUPS:
        print(f"{attr}, cut-off {cut} ({t:.4f})"); print(table_two(results, attr, cut).to_string(index=False)); print()

derived_race, cut-off 0.50 (0.5000)
                                    group     n accepts   TPR   FPR        read                 TPR gap                 FPR gap
                        White (reference) 66107   0.903 0.986 0.355         yes                                                
                       Race Not Available 17872   0.870 0.983 0.325         yes -0.003 [-0.006, -0.001] -0.030 [-0.050, -0.012]
                Black or African American  8991   0.796 0.973 0.296         yes -0.014 [-0.018, -0.010] -0.058 [-0.080, -0.038]
                                    Asian  8009   0.931 0.993 0.432         yes   +0.007 [0.004, 0.009]   +0.078 [0.043, 0.110]
                                    Joint  2716   0.904 0.986 0.337         yes  -0.000 [-0.005, 0.005]  -0.018 [-0.066, 0.031]
         American Indian or Alaska Native   810   0.753 0.964 0.190         yes -0.022 [-0.037, -0.008] -0.165 [-0.217, -0.111]
                 2 or more minority races   278                     

## 2. Fair models: what each step costs and what it changes

Every model is LightGBM with the same tuned parameters on the same frozen folds, so one thing changes at a time.

| list | with the demographic fields | race-blind | race-blind + reweighing |
|---|---|---|---|
| Kaggle (55 columns) | fair_full | ablate_no_demographics (38) | fair_full_reweigh (38) |
| clean (53 columns, 2 leaks removed) | fair_clean | fair_unaware (36) | fair_reweigh (36) |

- Race-blind: the 17 declared demographic fields are removed (race, ethnicity, sex and age of applicant and co-applicant, and their derived summaries).
- Reweighing (Kamiran and Calders, 2012): in the training part of each fold, each row gets the weight P(g) P(y) / P(g, y), with g = "derived_race" and y = accepted. In the weighted training data, race and the label are independent. Race only sets the weights and is not a feature; scoring is unweighted, as for every other run.

Each model is compared with the step before it: Kaggle list to clean list (the leaks), with the demographic fields to race-blind, race-blind to reweighed. AUC changes are paired per-fold differences with the corrected standard error of "make compare".

Gaps are other group minus reference group, at each model's own "same rate" cut-off (it accepts 85% of rows, as lenders did). The recorded decisions come first: they are what a perfectly faithful model would reproduce. The final blend is shown for reference. The bootstrap draws the same rows for every model, so the change of a gap between two models has a paired interval.

In [6]:
FAIR = {("Kaggle", "with demographics"): "fair_full",
        ("Kaggle", "race-blind"): "ablate_no_demographics",
        ("Kaggle", "race-blind + reweighing"): "fair_full_reweigh",
        ("clean", "with demographics"): "fair_clean",
        ("clean", "race-blind"): "fair_unaware",
        ("clean", "race-blind + reweighing"): "fair_reweigh"}
PREVIOUS = {"fair_clean": "fair_full", "ablate_no_demographics": "fair_full", "fair_unaware": "fair_clean",
            "fair_full_reweigh": "ablate_no_demographics", "fair_reweigh": "fair_unaware"}
SAME_AS_FULL = "20261003T084548Z_406e4210"   # lgbm_optuna_best: same config as fair_full, used if fair_full was not run


def latest(exp):
    r = runs[runs.experiment == f"configs/experiments/{exp}.yaml"]
    if len(r) == 0 and exp == "fair_full":
        r = runs[runs.run_id == SAME_AS_FULL]
    return r.iloc[-1] if len(r) else None


fair_run = {e: latest(e) for e in FAIR.values()}
missing = [e for e, r in fair_run.items() if r is None]
fair_run = {e: r for e, r in fair_run.items() if r is not None}
if missing:
    print("not run yet, left out of every table below:", ", ".join(missing))
assert len({r.folds_hash for r in fair_run.values()} | {runs.set_index("run_id").folds_hash[RUN_ID]}) == 1, "not the same folds"
fair_pred = {e: oof_pred(r.run_id)[0] for e, r in fair_run.items()}

t0 = time.perf_counter()
fair_results = {}
for e, pe in fair_pred.items():
    fair_results[e] = {attr: diagnostics(pe, attr) for attr in GROUPS}
    print(f"{e}: bootstrap done, {time.perf_counter() - t0:.0f}s", flush=True)

fair_full: bootstrap done, 60s
ablate_no_demographics: bootstrap done, 111s
fair_full_reweigh: bootstrap done, 162s
fair_clean: bootstrap done, 218s
fair_unaware: bootstrap done, 269s
fair_reweigh: bootstrap done, 317s


In [7]:
def name_of(e):
    lst, step = next(k for k, v in FAIR.items() if v == e)
    return f"{lst} list, {step}"


rows = []
for e, r in fair_run.items():
    row = {"model": name_of(e), "config": e, "columns": r.n_features, "CV AUC": f"{r.mean_auc:.5f}"}
    for label, base in (("vs step before", PREVIOUS.get(e)), ("vs fair_full", "fair_full")):
        if base in fair_run and base != e:
            f = fold_diff(fair_run[base], r)
            row[label] = f"{f['mean_diff']:+.5f} ({f['mean_diff'] / f['corrected_se']:+.1f} SE)"
    rows.append(row)
cost = pd.DataFrame(rows).fillna("")
print(cost.to_string(index=False))

                               model                 config  columns  CV AUC      vs step before        vs fair_full
      Kaggle list, with demographics              fair_full       55 0.91905                                        
             Kaggle list, race-blind ablate_no_demographics       38 0.91636  -0.00270 (-5.2 SE)  -0.00270 (-5.2 SE)
Kaggle list, race-blind + reweighing      fair_full_reweigh       38 0.91560  -0.00075 (-0.9 SE)  -0.00345 (-4.3 SE)
       clean list, with demographics             fair_clean       53 0.89305 -0.02600 (-13.6 SE) -0.02600 (-13.6 SE)
              clean list, race-blind           fair_unaware       36 0.89060  -0.00245 (-3.1 SE) -0.02846 (-16.0 SE)
 clean list, race-blind + reweighing           fair_reweigh       36 0.88985  -0.00075 (-2.4 SE) -0.02920 (-15.7 SE)


In [8]:
PAIRS = {"derived_race": "Black or African American", "derived_ethnicity": "Hispanic or Latino", "derived_sex": "Female"}
GAPS = {"accept same rate": "accept gap", "tpr same rate": "TPR gap", "fpr same rate": "FPR gap"}


def gap(res, attr, other, stat):
    """(point, draws) of other minus reference for one statistic."""
    point, draws, _ = res[attr]
    ref, k = GROUPS[attr], STATS.index(stat)
    return point[other][k] - point[GROUPS[attr]][k], draws[other][:, k] - draws[ref][:, k]


models = {"recorded decisions": None, "final blend": results} | {name_of(e): fair_results[e] for e in fair_results}
rows = []
for attr, other in PAIRS.items():
    for m, res in models.items():
        row = {"pair": f"{other} vs {GROUPS[attr]}", "model": m}
        if res is None:   # what lenders did: the observed accept rate
            pt, dr = gap(results, attr, other, "observed")
            row["accept gap"] = f"{pt:+.3f} {ci(dr)}"
        else:
            for s, name in GAPS.items():
                pt, dr = gap(res, attr, other, s)
                row[name] = f"{pt:+.3f} {ci(dr)}"
        rows.append(row)
gaps = pd.DataFrame(rows).fillna("")
print(gaps.to_string(index=False))

                                        pair                                model              accept gap                 TPR gap                 FPR gap
          Black or African American vs White                   recorded decisions -0.129 [-0.138, -0.120]                                                
          Black or African American vs White                          final blend -0.129 [-0.139, -0.119] -0.037 [-0.044, -0.030] -0.044 [-0.062, -0.025]
          Black or African American vs White       Kaggle list, with demographics -0.129 [-0.139, -0.119] -0.038 [-0.045, -0.031] -0.042 [-0.060, -0.023]
          Black or African American vs White              Kaggle list, race-blind -0.104 [-0.113, -0.095] -0.013 [-0.019, -0.007]  -0.017 [-0.037, 0.003]
          Black or African American vs White Kaggle list, race-blind + reweighing -0.098 [-0.107, -0.088] -0.008 [-0.013, -0.002]  -0.008 [-0.027, 0.012]
          Black or African American vs White        clean list, with demogra

## 2. Cost against gain, step by step

One row per step. AUC change as above. For each pair, the change of the accept gap at the same-rate cut-off, with its paired 95% interval: a positive change on a negative gap means the gap narrowed. In brackets, the share of the gap before the step that the step removes.

In [9]:
rows = []
for e in fair_results:
    base = PREVIOUS.get(e)
    if base not in fair_results:
        continue
    f = fold_diff(fair_run[base], fair_run[e])
    row = {"step": f"{name_of(base)} -> {name_of(e)}",
           "AUC change": f"{f['mean_diff']:+.5f} ({f['mean_diff'] / f['corrected_se']:+.1f} SE)"}
    for attr, other in PAIRS.items():
        before, d_before = gap(fair_results[base], attr, other, "accept same rate")
        after, d_after = gap(fair_results[e], attr, other, "accept same rate")
        row[f"{other} accept gap"] = f"{before:+.3f} -> {after:+.3f}: {after - before:+.3f} {ci(d_after - d_before)} ({1 - after / before:.0%})"
    rows.append(row)
gain = pd.DataFrame(rows)
print(gain.to_string(index=False))

                                                           step          AUC change            Black or African American accept gap                  Hispanic or Latino accept gap                              Female accept gap
      Kaggle list, with demographics -> Kaggle list, race-blind  -0.00270 (-5.2 SE)   -0.129 -> -0.104: +0.025 [0.021, 0.028] (19%)   -0.069 -> -0.065: +0.004 [0.002, 0.006] (6%) -0.048 -> -0.048: -0.000 [-0.002, 0.002] (-0%)
Kaggle list, race-blind -> Kaggle list, race-blind + reweighing  -0.00075 (-0.9 SE)    -0.104 -> -0.098: +0.006 [0.004, 0.009] (6%)  -0.065 -> -0.064: +0.001 [-0.001, 0.003] (2%)  -0.048 -> -0.048: +0.000 [-0.001, 0.002] (1%)
Kaggle list, with demographics -> clean list, with demographics -0.02600 (-13.6 SE) -0.129 -> -0.139: -0.010 [-0.015, -0.006] (-8%) -0.069 -> -0.071: -0.002 [-0.005, 0.001] (-3%) -0.048 -> -0.048: -0.000 [-0.003, 0.002] (-1%)
        clean list, with demographics -> clean list, race-blind  -0.00245 (-3.1 SE)   -0.139 -> 

## 2. Fair models, group by group

Every group read (n >= 500), every fair model side by side, at each model's same-rate cut-off: the accept gap, then the TPR and FPR gaps against the reference group, then calibration (mean prediction minus observed rate) and AUC inside the group. The full two tables of part 1, for any one model, are printed by the last cell of this part.

In [10]:
def side_by_side(cell, models):
    """One row per group read (reference groups left out for the gaps), one column per model."""
    rows = []
    for attr in GROUPS:
        point, _, size = results[attr]
        for level in point:
            if size[level][0] < MIN_N:
                continue
            row = {"field": SHORT[attr], "group": level, "n": size[level][0]}
            for m, res in models.items():
                row[m] = cell(res, attr, level)
            rows.append(row)
    return pd.DataFrame(rows)


def gap_cell(stat):
    def cell(res, attr, level):
        if level == GROUPS[attr]:
            return "reference"
        pt, dr = gap(res, attr, level, stat)
        return f"{pt:+.3f} {ci(dr)}"
    return cell


def own_cell(stat, sign=False):
    def cell(res, attr, level):
        point, draws, _ = res[attr]
        k = STATS.index(stat)
        return f"{point[level][k]:{'+' if sign else ''}.3f} {ci(draws[level][:, k])}"
    return cell


SHORT = {"derived_race": "race", "derived_ethnicity": "ethnicity", "derived_sex": "sex", "applicant_age": "age"}
COLS = {"Kaggle list, race-blind": "ablate_no_demographics", "Kaggle list, race-blind + reweighing": "fair_full_reweigh",
        "clean list, race-blind": "fair_unaware", "clean list, race-blind + reweighing": "fair_reweigh"}
side_models = {"final blend": results} | {f"{e}": fair_results[e] for e in COLS.values() if e in fair_results}
side = {"accept gap": side_by_side(gap_cell("accept same rate"), side_models),
        "TPR gap": side_by_side(gap_cell("tpr same rate"), side_models),
        "FPR gap": side_by_side(gap_cell("fpr same rate"), side_models),
        "pred - obs": side_by_side(own_cell("pred_minus_obs", sign=True), side_models),
        "AUC": side_by_side(own_cell("auc"), side_models)}
for k, t in side.items():
    print(k); print(t.to_string(index=False)); print()

accept gap
    field                            group     n             final blend  ablate_no_demographics       fair_full_reweigh            fair_unaware            fair_reweigh
     race                            White 66107               reference               reference               reference               reference               reference
     race               Race Not Available 17872 -0.044 [-0.050, -0.038] -0.035 [-0.041, -0.029] -0.033 [-0.039, -0.027] -0.035 [-0.041, -0.029] -0.031 [-0.037, -0.025]
     race        Black or African American  8991 -0.129 [-0.139, -0.119] -0.104 [-0.113, -0.095] -0.098 [-0.107, -0.088] -0.108 [-0.117, -0.099] -0.096 [-0.104, -0.087]
     race                            Asian  8009   +0.038 [0.031, 0.045]   +0.041 [0.034, 0.048]   +0.041 [0.035, 0.048]   +0.048 [0.041, 0.055]   +0.048 [0.041, 0.054]
     race                            Joint  2716  +0.005 [-0.008, 0.017]  +0.009 [-0.004, 0.022]  +0.012 [-0.001, 0.024]  +0.011 [-0.003, 0.023]

In [11]:
ONE = "fair_reweigh"   # any config of FAIR: its two full tables, as for the competition model
res_one = fair_results[ONE]
print(ONE, fair_run[ONE].run_id, "  cut-offs", {k: round(v, 4) for k, v in cut_offs(fair_pred[ONE]).items()})
for attr in GROUPS:
    print(attr); print(table_one(res_one, attr).to_string(index=False)); print()
for attr in GROUPS:
    print(f"{attr}, cut-off same rate"); print(table_two(res_one, attr, "same rate").to_string(index=False)); print()

fair_reweigh 20261109T183339Z_9ddab9a2   cut-offs {'0.50': 0.5, 'same rate': 0.7879}
derived_race
                                    group     n  denied             observed predicted              pred - obs                  AUC AUC without leak rows        read
                                    White 66107    8721 0.868 [0.865, 0.871]     0.861 -0.007 [-0.009, -0.006] 0.891 [0.886, 0.895]  0.903 [0.898, 0.907]         yes
                       Race Not Available 17872    3082 0.828 [0.822, 0.833]     0.840   +0.013 [0.009, 0.016] 0.890 [0.883, 0.898]  0.904 [0.896, 0.912]         yes
                Black or African American  8991    2349 0.739 [0.730, 0.748]     0.776   +0.038 [0.031, 0.044] 0.885 [0.875, 0.893]  0.892 [0.883, 0.901]         yes
                                    Asian  8009     888 0.889 [0.882, 0.896]     0.893  +0.004 [-0.001, 0.010] 0.844 [0.827, 0.860]  0.849 [0.830, 0.866]         yes
                                    Joint  2716     347 0.872 [0.859, 0.

## 3. Proxy check: can the remaining columns tell an applicant's race?

Removing the demographic fields only helps if the other columns do not carry the same information. "src/proxy_check.py" trains, for each race group with at least 5,000 rows, a LightGBM that predicts "this applicant is in the group" against everyone else, out-of-fold on the frozen folds, with the competition settings. Once from all the columns of the race-blind Kaggle list, once from the geography columns alone. AUC 0.5 = the columns say nothing about that group.

In [21]:
proxy_path = ROOT / "reports" / "tuning" / "proxy_check.csv"
if proxy_path.exists():
    proxy_table = pd.read_csv(proxy_path)
    print(proxy_table.round(4).to_string(index=False))
else:
    proxy_table = None
    print("not run yet: uv run python src/proxy_check.py configs/experiments/ablate_no_demographics.yaml")

                    group  rows       columns  mean fold AUC    min    max
                    White 66107      all (38)         0.7401 0.7353 0.7427
                    White 66107 geography (8)         0.6949 0.6893 0.6995
       Race Not Available 17872      all (38)         0.6717 0.6693 0.6748
       Race Not Available 17872 geography (8)         0.5871 0.5809 0.5915
Black or African American  8991      all (38)         0.8398 0.8328 0.8488
Black or African American  8991 geography (8)         0.7958 0.7903 0.8003
                    Asian  8009      all (38)         0.8465 0.8398 0.8508
                    Asian  8009 geography (8)         0.8003 0.7919 0.8046


## Tables for the report

The cell below prints the tables of "reports/fairness.md" in markdown, so the report is copied from here and never typed by hand. Their reading is in the report.

In [22]:
def markdown(table):
    lines = ["| " + " | ".join(table.columns) + " |", "|" + "---|" * len(table.columns)]
    return "\n".join(lines + ["| " + " | ".join(str(v) for v in row) + " |" for row in table.to_numpy()])


def report_table(make, drop):
    t = pd.concat([make(attr).assign(field=SHORT[attr]) for attr in GROUPS]).fillna("")
    first = next(c for c in t.columns if c not in ("group", "n", "denied", "read", "field"))
    t[first] = t[first].where(t.read == "yes", "not read")
    return t[["field"] + [c for c in t.columns if c not in ("field", "read", *drop)]]


print("<competition: calibration>\n" + markdown(report_table(lambda a: table_one(results, a), ["predicted"])))
for cut in ("same rate", "0.50"):
    print(f"\n<competition: cut {cut}>\n" + markdown(report_table(lambda a: table_two(results, a, cut), [])))
print("\n<fair: cost>\n" + markdown(cost))
print("\n<fair: gaps>\n" + markdown(gaps))
print("\n<fair: cost against gain>\n" + markdown(gain))
for k, t in side.items():
    print(f"\n<fair side by side: {k}>\n" + markdown(t))
if proxy_table is not None:
    print("\n<proxy>\n" + markdown(proxy_table.round(4)))

<competition: calibration>
| field | group | n | denied | observed | pred - obs | AUC | AUC without leak rows |
|---|---|---|---|---|---|---|---|
| race | White | 66107 | 8721 | 0.868 [0.865, 0.871] | +0.001 [-0.001, 0.002] | 0.921 [0.917, 0.924] | 0.907 [0.903, 0.911] |
| race | Race Not Available | 17872 | 3082 | 0.828 [0.822, 0.833] | +0.003 [-0.001, 0.006] | 0.923 [0.916, 0.929] | 0.909 [0.901, 0.916] |
| race | Black or African American | 8991 | 2349 | 0.739 [0.730, 0.748] | +0.005 [-0.001, 0.011] | 0.909 [0.901, 0.917] | 0.896 [0.886, 0.904] |
| race | Asian | 8009 | 888 | 0.889 [0.882, 0.896] | +0.004 [-0.001, 0.008] | 0.882 [0.867, 0.896] | 0.855 [0.837, 0.872] |
| race | Joint | 2716 | 347 | 0.872 [0.859, 0.884] | -0.001 [-0.009, 0.007] | 0.911 [0.889, 0.933] | 0.895 [0.870, 0.921] |
| race | American Indian or Alaska Native | 810 | 221 | 0.727 [0.696, 0.758] | -0.008 [-0.026, 0.010] | 0.939 [0.917, 0.960] | 0.933 [0.908, 0.956] |
| race | 2 or more minority races | 278 | 87 |

## Test: does removing the leaks widen the Black / White gap through the leak rows?

Hypothesis: the gap widens from the Kaggle list to the clean list because of the 6,279 leak rows. There the Kaggle-list model reads the decision from the leak column, so race plays no part in its score; the clean model must score those rows from the other columns, the demographic fields included.

Prediction if it holds: on the rows without a leak value, the gap barely changes (the interval of the change includes 0); on the leak rows, it widens.

Test: the same out-of-fold predictions, read on each subset of rows. Each model keeps its same-rate cut-off set on all rows, as in the tables above. Paired bootstrap over rows inside each group (1000 resamples). Needs only the first code cell.

In [23]:
def accept_gap_test():
    leak_rows = ((df.preapproval.astype("string") == "1")
                 | (df.initially_payable_to_institution.astype("string") == "3")).to_numpy()
    race = df.derived_race.astype("string").to_numpy()
    white, black = race == "White", race == "Black or African American"
    full_id = runs[runs.experiment == "configs/experiments/fair_full.yaml"].run_id
    full_id = full_id.iloc[-1] if len(full_id) else "20261003T084548Z_406e4210"   # same config as fair_full
    clean_id = runs[runs.experiment == "configs/experiments/fair_clean.yaml"].run_id.iloc[-1]
    accepts = {}
    for name, run_id in (("Kaggle list", full_id), ("clean list", clean_id)):
        pm = oof_pred(run_id)[0]
        accepts[name] = pm >= np.quantile(pm, 1 - y.mean())
    rng = np.random.default_rng(BOOT_SEED)
    print(f"leak rows {leak_rows.sum()}; share of leak rows: White {leak_rows[white].mean():.3f}, Black {leak_rows[black].mean():.3f}")
    rows = []
    for subset, sel in (("all", np.ones(len(y), bool)), ("without a leak value", ~leak_rows), ("leak rows only", leak_rows)):
        iw, ib = np.flatnonzero(white & sel), np.flatnonzero(black & sel)
        gap_of = lambda a, iw, ib: a[ib].mean() - a[iw].mean()
        g = {m: gap_of(a, iw, ib) for m, a in accepts.items()}
        change = []
        for _ in range(N_BOOT):
            bw, bb = rng.choice(iw, len(iw)), rng.choice(ib, len(ib))
            change.append(gap_of(accepts["clean list"], bw, bb) - gap_of(accepts["Kaggle list"], bw, bb))
        rows.append({"rows": subset, "White / Black rows": f"{len(iw):,} / {len(ib):,}",
                     "accept gap, Kaggle list": f"{g['Kaggle list']:+.3f}", "accept gap, clean list": f"{g['clean list']:+.3f}",
                     "change [95%]": f"{g['clean list'] - g['Kaggle list']:+.3f} {ci(np.array(change))}"})
    return pd.DataFrame(rows)


def ci(a, digits=3):   # same as in part 1, repeated so this cell runs after the first code cell alone
    lo, hi = np.nanquantile(a, [0.025, 0.975])
    return f"[{lo:.{digits}f}, {hi:.{digits}f}]"


leak_test = accept_gap_test()
print(leak_test.to_string(index=False))
lines = ["| " + " | ".join(leak_test.columns) + " |", "|" + "---|" * len(leak_test.columns)]
print("\n<leak rows test>\n" + "\n".join(lines + ["| " + " | ".join(map(str, r)) + " |" for r in leak_test.to_numpy()]))

leak rows 6279; share of leak rows: White 0.060, Black 0.059
                rows White / Black rows accept gap, Kaggle list accept gap, clean list            change [95%]
                 all     66,107 / 8,991                  -0.129                 -0.139 -0.010 [-0.015, -0.006]
without a leak value     62,128 / 8,465                  -0.126                 -0.140 -0.014 [-0.018, -0.010]
      leak rows only        3,979 / 526                  -0.179                 -0.125   +0.054 [0.013, 0.093]

<leak rows test>
| rows | White / Black rows | accept gap, Kaggle list | accept gap, clean list | change [95%] |
|---|---|---|---|---|
| all | 66,107 / 8,991 | -0.129 | -0.139 | -0.010 [-0.015, -0.006] |
| without a leak value | 62,128 / 8,465 | -0.126 | -0.140 | -0.014 [-0.018, -0.010] |
| leak rows only | 3,979 / 526 | -0.179 | -0.125 | +0.054 [0.013, 0.093] |


## 4. Which fairness criterion: the conflict in numbers

Three criteria, with accepted as the positive class, between White and Black or African American applicants:

- equal accept rates: the model accepts the same share of each group;
- equal error rates: the same TPR (share of applicants lenders accepted that the model accepts) and the same FPR (share of applicants lenders denied that the model accepts);
- calibration within groups: a given score means the same accept rate in every group.

At a cut-off, for any group, FPR = p / (1 - p) x (1 - PPV) / PPV x TPR, where p is the group's accept rate in the recorded decisions and PPV the share of the model's acceptances that lenders also accepted (Chouldechova, 2017). The first table checks the identity on our models. The second reads calibration inside score bands: the five quintiles of each model's score, pooled over all rows. The last cell describes the "observed" fields. Needs only the first code cell.

In [24]:
def md_table(t):
    lines = ["| " + " | ".join(t.columns) + " |", "|" + "---|" * len(t.columns)]
    return "\n".join(lines + ["| " + " | ".join(map(str, r)) + " |" for r in t.to_numpy()])


latest_id = lambda e: runs[runs.experiment == f"configs/experiments/{e}.yaml"].run_id.iloc[-1]
CRIT_MODELS = {"final blend": RUN_ID, "fair_full_reweigh": latest_id("fair_full_reweigh"), "fair_reweigh": latest_id("fair_reweigh")}
race_c = df.derived_race.astype("string").to_numpy()
CRIT_GROUPS = {"White": race_c == "White", "Black or African American": race_c == "Black or African American"}
crit_pred = {m: oof_pred(r)[0] for m, r in CRIT_MODELS.items()}

rows = []
for m, pm in crit_pred.items():
    acc_m = pm >= np.quantile(pm, 1 - y.mean())
    for g, s in CRIT_GROUPS.items():
        yy, aa = y[s], acc_m[s]
        base, tpr, fpr, ppv = yy.mean(), aa[yy == 1].mean(), aa[yy == 0].mean(), yy[aa].mean()
        identity = base / (1 - base) * (1 - ppv) / ppv * tpr
        assert abs(identity - fpr) < 1e-9
        rows.append({"model": m, "group": g, "recorded accept rate": f"{base:.3f}", "mean prediction": f"{pm[s].mean():.3f}",
                     "model accepts": f"{aa.mean():.3f}", "TPR": f"{tpr:.3f}", "FPR": f"{fpr:.3f}", "PPV": f"{ppv:.3f}"})
crit_table = pd.DataFrame(rows)
pw, pb = (y[s].mean() for s in CRIT_GROUPS.values())
print("identity checked for every row; equal PPV and TPR would force FPR(Black) / FPR(White) =",
      round((pb / (1 - pb)) / (pw / (1 - pw)), 3))
print(crit_table.to_string(index=False))

band_rows = []
for m in ("final blend", "fair_reweigh"):
    pm = crit_pred[m]
    band = pd.qcut(pm, 5, labels=False)
    for b in range(5):
        row = {"model": m, "score band": f"{b + 1}: {pm[band == b].min():.2f} to {pm[band == b].max():.2f}"}
        for g, s in CRIT_GROUPS.items():
            k = s & (band == b)
            short = "White" if g == "White" else "Black"
            row[f"{short} n"] = int(k.sum())
            row[f"{short}: mean prediction / recorded"] = f"{pm[k].mean():.3f} / {y[k].mean():.3f}"
        band_rows.append(row)
band_table = pd.DataFrame(band_rows)
print(band_table.to_string(index=False))
print("\n<criteria: error rates>\n" + md_table(crit_table) + "\n\n<criteria: score bands>\n" + md_table(band_table))

identity checked for every row; equal PPV and TPR would force FPR(Black) / FPR(White) = 0.43
            model                     group recorded accept rate mean prediction model accepts   TPR   FPR   PPV
      final blend                     White                0.868           0.869         0.868 0.959 0.264 0.960
      final blend Black or African American                0.739           0.744         0.739 0.922 0.220 0.922
fair_full_reweigh                     White                0.868           0.862         0.862 0.954 0.258 0.960
fair_full_reweigh Black or African American                0.739           0.774         0.765 0.947 0.250 0.914
     fair_reweigh                     White                0.868           0.861         0.861 0.945 0.311 0.952
     fair_reweigh Black or African American                0.739           0.776         0.766 0.935 0.288 0.902
       model      score band  White n White: mean prediction / recorded  Black n Black: mean prediction / recorded
 

In [25]:
# The "observed" fields: 1 = recorded by the lender from visual observation or surname, 2 = not recorded on that basis, 3 = not applicable
obs_rows = []
for c in ("applicant_race_observed", "applicant_ethnicity_observed", "applicant_sex_observed"):
    v = df[c].astype("string")
    for level in ("1", "2", "3"):
        obs_rows.append({"field": c, "value": level, "rows": int((v == level).sum()), "accepted": f"{y[(v == level).to_numpy()].mean():.3f}"})
obs_table = pd.DataFrame(obs_rows)
by_race = pd.crosstab(df.derived_race.astype("string"), df.applicant_race_observed.astype("string"))
print(obs_table.to_string(index=False)); print(by_race.to_string())
print("\n<criteria: observed fields>\n" + md_table(obs_table))

                       field value  rows accepted
     applicant_race_observed     1  1320    0.855
     applicant_race_observed     2 97421    0.849
     applicant_race_observed     3  6259    0.863
applicant_ethnicity_observed     1  1305    0.856
applicant_ethnicity_observed     2 97455    0.849
applicant_ethnicity_observed     3  6240    0.864
      applicant_sex_observed     1  1306    0.861
      applicant_sex_observed     2 97401    0.849
      applicant_sex_observed     3  6293    0.863
applicant_race_observed                       1      2     3
derived_race                                                
2 or more minority races                      3    269     6
American Indian or Alaska Native              1    784    25
Asian                                        74   7736   199
Black or African American                   122   8550   319
Free Form Text Only                           0     21     0
Joint                                        26   2619    71
Native Hawai

## Test: does the remaining gap come through geography?

We think that part of the Black-White gap left by the fair model (fair_reweigh: accept gap -0.096, TPR gap -0.010) comes through the geography columns, which identify Black applicants with AUC 0.796 on their own. If so, the same model without the 8 geography columns (fair_reweigh_no_geo: state, the area's median family income, the 6 census tract columns; 28 columns) should have a smaller gap.

The rule, fixed before the run: the hypothesis holds if the accept gap narrows with a paired 95% bootstrap interval above 0, and is rejected otherwise. Each model is read at its own same-rate cut-off; the bootstrap draws the same White and Black rows for both models (1,000 resamples). Run the first code cell, then this one.

In [ ]:
def geo_test(old="fair_reweigh", new="fair_reweigh_no_geo"):
    """Black - White gaps of two models at their own same-rate cut-off, with a paired bootstrap of the change."""
    race = df.derived_race.astype("string").to_numpy()
    iw, ib = np.flatnonzero(race == "White"), np.flatnonzero(race == "Black or African American")
    models = {}
    for exp in (old, new):
        r = runs[runs.experiment == f"configs/experiments/{exp}.yaml"]
        assert len(r), f"{exp} has not been run yet"
        r = r.iloc[-1]
        pm = oof_pred(r.run_id)[0]
        models[exp] = (r, pm, (pm >= np.quantile(pm, 1 - y.mean())).astype(float))
    assert models[old][0].folds_hash == models[new][0].folds_hash, "not the same folds"

    def stats(pm, a, w, b):
        def one(i):
            pos = y[i] == 1
            return np.array([a[i].mean(), a[i][pos].mean(), a[i][~pos].mean(), pm[i].mean() - y[i].mean()])
        sw, sb = one(w), one(b)
        return np.r_[sb[:3] - sw[:3], sb[3], sw[3]]

    names = ["accept gap", "TPR gap", "FPR gap", "pred - obs, Black", "pred - obs, White"]
    point = {e: stats(pm, a, iw, ib) for e, (_, pm, a) in models.items()}
    rng = np.random.default_rng(BOOT_SEED)
    change = []
    for _ in range(N_BOOT):
        bw, bb = rng.choice(iw, len(iw)), rng.choice(ib, len(ib))
        change.append(stats(*models[new][1:], bw, bb) - stats(*models[old][1:], bw, bb))
    change = np.array(change)
    for e, (r, _, _) in models.items():
        print(f"{e}: run {r.run_id}, {r.n_features} columns, CV AUC {r.mean_auc:.5f}")
    return pd.DataFrame({"statistic": names, old: [f"{v:+.3f}" for v in point[old]],
                         new: [f"{v:+.3f}" for v in point[new]],
                         "change [95%]": [f"{point[new][k] - point[old][k]:+.3f} {ci(change[:, k])}" for k in range(len(names))]})


def ci(a, digits=3):   # same as in part 1, repeated so this cell runs after the first code cell alone
    lo, hi = np.nanquantile(a, [0.025, 0.975])
    return f"[{lo:.{digits}f}, {hi:.{digits}f}]"


geo = geo_test()
print(geo.to_string(index=False))
lines = ["| " + " | ".join(geo.columns) + " |", "|" + "---|" * len(geo.columns)]
print("\n<geography test>\n" + "\n".join(lines + ["| " + " | ".join(map(str, r)) + " |" for r in geo.to_numpy()]))